# 📖 Notebook 2: Range-Based Sharding

Range-based sharding is the most **intuitive** sharding strategy. Instead of hashing, you split data by **value ranges**. For example:

```
Shard 0 → User IDs    1 – 100
Shard 1 → User IDs  101 – 200
Shard 2 → User IDs  201 – 300
```

This is simple, predictable, and supports efficient **range scans**. But it has a big weakness: **uneven distribution** and **hot spots**.

## Learning Objectives

By the end of this notebook, you'll understand:
- How range-based sharding works
- When range sharding is a good fit (and when it's not)
- The hot spot problem with time-based range keys
- How to compare range vs hash sharding distribution

## 🛠️ Setup

Make sure infrastructure is running:

```bash
cd core-concepts/sharding
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import random
import time
from datetime import datetime, timedelta

SHARD_CONFIGS = {
    0: {"host": "localhost", "port": 5433, "database": "shard_1", "user": "demo", "password": "demo"},
    1: {"host": "localhost", "port": 5434, "database": "shard_2", "user": "demo", "password": "demo"},
    2: {"host": "localhost", "port": 5435, "database": "shard_3", "user": "demo", "password": "demo"},
}

NUM_SHARDS = len(SHARD_CONFIGS)

def get_connection(shard_id):
    return psycopg2.connect(**SHARD_CONFIGS[shard_id])

# Verify connections
for sid in SHARD_CONFIGS:
    conn = get_connection(sid)
    cur = conn.cursor()
    cur.execute("SELECT current_database()")
    print(f"✅ Shard {sid} → {cur.fetchone()[0]}")
    cur.close()
    conn.close()

## Step 1: Range-Based Shard Router

With range sharding, we define explicit boundaries for each shard. The router checks which range a key falls into.

In [ ]:
class RangeShardRouter:
    """
    Routes data to shards based on value ranges.
    
    Each range is defined as (min_value, max_value, shard_id).
    A key is routed to the shard whose range contains it.
    """
    
    def __init__(self, ranges, shard_configs):
        # ranges: list of (min_val, max_val, shard_id)
        self.ranges = sorted(ranges, key=lambda r: r[0])
        self.shard_configs = shard_configs
    
    def get_shard(self, key):
        """Find which shard a key belongs to by checking ranges."""
        for min_val, max_val, shard_id in self.ranges:
            if min_val <= key <= max_val:
                return shard_id
        raise ValueError(f"Key {key} doesn't fall in any range!")
    
    def get_connection(self, shard_id):
        return psycopg2.connect(**self.shard_configs[shard_id])
    
    def execute_on_shard(self, key, query, params=None):
        shard_id = self.get_shard(key)
        conn = self.get_connection(shard_id)
        conn.autocommit = True
        cur = conn.cursor()
        cur.execute(query, params)
        result = cur.fetchall() if cur.description else None
        cur.close()
        conn.close()
        return shard_id, result

# Define ranges: split 300 users evenly across 3 shards
ranges = [
    (1, 100, 0),    # Shard 0: user IDs 1–100
    (101, 200, 1),  # Shard 1: user IDs 101–200
    (201, 300, 2),  # Shard 2: user IDs 201–300
]

range_router = RangeShardRouter(ranges, SHARD_CONFIGS)

# Test routing
for user_id in [1, 50, 100, 101, 150, 201, 250, 300]:
    shard = range_router.get_shard(user_id)
    print(f"User {user_id:3d} → Shard {shard}")

## Step 2: Insert Data with Range Sharding

In [ ]:
COUNTRIES = ['US', 'UK', 'Germany', 'Japan', 'Brazil', 'India', 'Canada', 'France']

# Clear all shards
for sid in range(NUM_SHARDS):
    conn = get_connection(sid)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute("DELETE FROM orders")
    cur.execute("DELETE FROM users")
    cur.close()
    conn.close()

# Insert 300 users using range-based routing
shard_counts = {i: 0 for i in range(NUM_SHARDS)}

for user_id in range(1, 301):
    shard_id = range_router.get_shard(user_id)
    shard_counts[shard_id] += 1
    conn = range_router.get_connection(shard_id)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO users (id, username, email, country) VALUES (%s, %s, %s, %s)",
        (user_id, f"user_{user_id}", f"user{user_id}@example.com", random.choice(COUNTRIES))
    )
    cur.close()
    conn.close()

print("Range-based distribution:")
print("─" * 40)
for sid, count in shard_counts.items():
    bar = '█' * (count // 3)
    lo, hi, _ = ranges[sid]
    print(f"Shard {sid} (IDs {lo:3d}–{hi:3d}): {count:3d} users {bar}")
print()
print("✅ Perfectly balanced! Each shard gets exactly 100 users.")
print("   This is the ideal case where IDs are sequential and evenly distributed.")

## Step 3: Range Queries — Where Range Sharding Shines ✨

The killer feature of range sharding is efficient **range scans**. Need all users with IDs 50–75? That's a single shard query.

In [ ]:
# Range query: get users 50–75 (all on Shard 0)
start = time.time()

shard_id = range_router.get_shard(50)  # both 50 and 75 are on shard 0
conn = range_router.get_connection(shard_id)
cur = conn.cursor()
cur.execute("SELECT id, username FROM users WHERE id BETWEEN 50 AND 75 ORDER BY id")
results = cur.fetchall()
elapsed = (time.time() - start) * 1000
cur.close()
conn.close()

print(f"🔍 Range query: users 50–75")
print(f"   Hit only Shard {shard_id} (all users in this range live there)")
print(f"   Found {len(results)} users in {elapsed:.1f} ms")
print()

# Now try a range query that SPANS shards: users 90–110
start = time.time()
shard_a = range_router.get_shard(90)   # Shard 0
shard_b = range_router.get_shard(110)  # Shard 1

all_results = []
for sid in set([shard_a, shard_b]):
    conn = range_router.get_connection(sid)
    cur = conn.cursor()
    cur.execute("SELECT id, username FROM users WHERE id BETWEEN 90 AND 110 ORDER BY id")
    all_results.extend(cur.fetchall())
    cur.close()
    conn.close()

elapsed = (time.time() - start) * 1000

print(f"🔍 Range query: users 90–110 (crosses shard boundary!)")
print(f"   Had to query Shard {shard_a} AND Shard {shard_b}")
print(f"   Found {len(all_results)} users in {elapsed:.1f} ms")

## Step 4: The Hot Spot Problem 🔥

Range sharding looks great when IDs are evenly distributed. But what about **time-based** keys?

Imagine sharding orders by `created_at`. All **new** orders go to the latest shard while old shards sit idle. This creates a massive hot spot.

In [ ]:
# Simulate time-based range sharding for orders
# Shard 0: Jan–Feb, Shard 1: Mar–Apr, Shard 2: May–Jun
from datetime import datetime, timedelta

def get_time_shard(created_at):
    """Shard by month: 0=Jan-Feb, 1=Mar-Apr, 2=May-Jun."""
    month = created_at.month
    if month <= 2:
        return 0
    elif month <= 4:
        return 1
    else:
        return 2

# Simulate 600 orders created over 6 months
# BUT most activity is recent (last 2 months = shard 2)
order_distribution = {0: 0, 1: 0, 2: 0}
write_distribution = {0: 0, 1: 0, 2: 0}

base_date = datetime(2024, 1, 1)
for i in range(600):
    # Simulate real-world pattern: 70% of orders are in the last 2 months
    if random.random() < 0.7:
        days_offset = random.randint(120, 180)  # May–Jun
    elif random.random() < 0.5:
        days_offset = random.randint(60, 120)   # Mar–Apr
    else:
        days_offset = random.randint(0, 60)     # Jan–Feb
    
    order_date = base_date + timedelta(days=days_offset)
    shard = get_time_shard(order_date)
    order_distribution[shard] += 1

# New orders (writes) ALL go to the latest shard
now = datetime(2024, 6, 15)  # It's June now
for i in range(100):
    shard = get_time_shard(now)
    write_distribution[shard] += 1

print("🔥 Time-based range sharding — the hot spot problem")
print("═" * 55)
print()
print("Data distribution (total orders per shard):")
print("─" * 55)
for sid, count in order_distribution.items():
    months = ['Jan–Feb', 'Mar–Apr', 'May–Jun'][sid]
    bar = '█' * (count // 10)
    print(f"  Shard {sid} ({months}): {count:3d} orders {bar}")

print()
print("Write traffic (100 new orders placed TODAY):")
print("─" * 55)
for sid, count in write_distribution.items():
    months = ['Jan–Feb', 'Mar–Apr', 'May–Jun'][sid]
    bar = '🔴' * (count // 5) if count > 0 else ''
    print(f"  Shard {sid} ({months}): {count:3d} writes {bar}")

print()
print("⚠️  ALL writes go to Shard 2 (the newest time range).")
print("   Shards 0 and 1 handle almost no traffic — they're wasted capacity.")
print("   This is why time-based range sharding creates hot spots.")

## Step 5: Hash vs Range — Side by Side

Let's compare both strategies directly on the same dataset.

In [ ]:
import hashlib

def hash_shard(key, num_shards):
    key_bytes = str(key).encode('utf-8')
    hash_digest = hashlib.md5(key_bytes).hexdigest()
    return int(hash_digest[:8], 16) % num_shards

# 10,000 user IDs — compare distribution with both strategies
n_users = 10000
n_shards = 3

# Range-based: IDs 1–3333 → Shard 0, 3334–6666 → Shard 1, 6667–10000 → Shard 2
range_counts = {0: 0, 1: 0, 2: 0}
hash_counts = {0: 0, 1: 0, 2: 0}

for uid in range(1, n_users + 1):
    # Range
    if uid <= 3333:
        range_counts[0] += 1
    elif uid <= 6666:
        range_counts[1] += 1
    else:
        range_counts[2] += 1
    
    # Hash
    hash_counts[hash_shard(uid, n_shards)] += 1

print("📊 Distribution comparison (10,000 users, 3 shards)")
print("═" * 55)
print()
print("Range-based sharding:")
for s, c in range_counts.items():
    pct = c / n_users * 100
    bar = '█' * int(pct / 2)
    print(f"  Shard {s}: {c:5d} ({pct:.1f}%) {bar}")

print()
print("Hash-based sharding:")
for s, c in hash_counts.items():
    pct = c / n_users * 100
    bar = '█' * int(pct / 2)
    print(f"  Shard {s}: {c:5d} ({pct:.1f}%) {bar}")

print()
print("Both distribute evenly for sequential IDs.")
print("The difference shows up with ACCESS PATTERNS, not just data placement.")
print("")
print("┌──────────────────┬──────────────────────────┬────────────────────────────┐")
print("│ Feature          │ Range Sharding           │ Hash Sharding              │")
print("├──────────────────┼──────────────────────────┼────────────────────────────┤")
print("│ Range queries    │ ✅ Efficient (one shard) │ ❌ Must hit all shards     │")
print("│ Even writes      │ ❌ Risk of hot spots     │ ✅ Spread evenly           │")
print("│ Add shards       │ ✅ Just split a range    │ ❌ Rehash everything       │")
print("│ Predictable      │ ✅ Easy to understand    │ ❌ Hard to predict         │")
print("└──────────────────┴──────────────────────────┴────────────────────────────┘")

## Step 6: When Range Sharding Works Well

Range sharding is a great fit for **multi-tenant systems** where each tenant naturally queries their own data.

In [ ]:
# Multi-tenant SaaS: each company gets a range of user IDs
# Company A: IDs 1–100, Company B: IDs 101–200, Company C: IDs 201–300

tenants = {
    'Acme Corp':     (1, 100, 0),
    'Beta Inc':      (101, 200, 1),
    'Charlie LLC':   (201, 300, 2),
}

# Simulate queries — each company only queries its own users
print("🏢 Multi-tenant SaaS — range sharding by tenant")
print("═" * 55)
print()

for company, (lo, hi, shard) in tenants.items():
    # Company's admin queries their own users
    conn = get_connection(shard)
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM users WHERE id BETWEEN %s AND %s", (lo, hi))
    count = cur.fetchone()[0]
    cur.close()
    conn.close()
    print(f"  {company:15s} → Shard {shard} (IDs {lo}–{hi}) → {count} users")

print()
print("Each company's queries hit ONE shard only — no cross-shard traffic!")
print("This is the ideal use case for range-based sharding.")

## 🎯 Key Takeaways

1. **Range sharding** splits data by value ranges — simple and predictable
2. **Range queries are efficient** — data you need is often on one shard
3. **Hot spots are the main risk** — especially with time-based keys where all writes hit the newest shard
4. **Great for multi-tenant systems** — each tenant naturally queries their own range
5. **Adding shards is easier** — split one range into two, no need to rehash everything

### Next Up

**Notebook 3: Consistent Hashing** — How to add and remove shards without moving all the data.